In [2]:
import numpy as np
from tdmclient import ClientAsync
import cv2

from src.local_navigation.navigation_control import *
#from src.local_navigation.local_occupancy import *

from src.computer_vision.vision import Vision 
from src.global_navigation.global_nav import global_nav
from src.pose_estimation.extended_kalman_filter import ExtendedKalmanFilter
from src.pose_estimation.differential_drive_system import DifferentialDriveSystem
# Progress bar for loops visualization
from tqdm import tqdm
# Plotting
import matplotlib.pyplot as plt
from src.utils.plot_pose_estimation import *
%matplotlib inline


In [3]:
import cv2
import time

vision = Vision()

# 1) Open the webcam and capture ONE reference image
cap = cv2.VideoCapture(0)  # use index 0 because this is the one that works for you

if not cap.isOpened():
    raise RuntimeError("Failed to open webcam (index 1).")

time.sleep(1)  # wait for the camera to stabilize
ret, frame0 = cap.read()
if not ret or frame0 is None:
    cap.release()
    raise RuntimeError("Failed to capture a frame from the webcam.")

# We can close the camera now; the reference image is stored in frame0
cap.release()

# Just to verify that we really have an image
print("[DEBUG] Captured frame shape:", frame0.shape)
#cv2.imshow("Reference frame", frame0)
cv2.waitKey(1)  # small delay so the window can appear at least once

# 2) Initialize your Vision pipeline with the reference frame
print("[DEBUG] Calling vision.initialize(...)")
vision.initialize(frame0)  # e.g., color detection, blur, Canny, polygons, etc.

# 3) Process the same frame to extract obstacles and robot state
print("[DEBUG] Calling vision.process(...)")
obstacles, robot_state = vision.process(
    frame0,
    show_debug=False,    # no debug display from inside Vision
    skip_blur=False,
    log_pose=False
)

print("[DEBUG] Robot state:", robot_state)

# 4) Global navigation using the detected obstacles
robot_radius = 30
imgh, imgw = frame0.shape[:2]

print("[DEBUG] Calling global_nav(...)")
path = global_nav(vision, robot_radius, imgw, imgh, frame0)

print("[DEBUG] global_nav finished, path length:",
      len(path) if path is not None else None)

cv2.destroyAllWindows()


[DEBUG] Captured frame shape: (1080, 1920, 3)
[DEBUG] Calling vision.initialize(...)
[INFO] Initialisation terminé ( couleurs + Canny + polygones figé).
[DEBUG] Calling vision.process(...)
[DEBUG] Robot state: {'found': True, 'center': (106, 532), 'theta': 0.029841884667109353, 'red_center': (107, 499), 'green_center': (105, 566), 'robot_mask': array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=uint8), 'smooth_center': (106, 532), 'smooth_theta': 0.02984188466710936}
[DEBUG] Calling global_nav(...)
[GLOBAL] Robot trouvé avec postion valide, centre = (106, 532) theta(deg) = 1.7098140441415295
[GLOBAL] Goal valide, coordonnées = (1650, 1030)


100%|██████████| 3/3 [00:00<00:00, 856.10it/s]

Chemin trouvé :
   (106.0, 532.0)
   (948.5163866275714, 739.4417159923023)
   (1650.0, 1030.0)
[DEBUG] global_nav finished, path length: 3


In [21]:
cv2.destroyAllWindows()

In [4]:
# 1.6 EKF + Thymio client are created elsewhere, e.g.
# ekf = YourEKF(...)
# thymio = YourThymioClient(...)
lambda_ = 0.39735099337748336  # conversion factor
d = 93.5         # distance between wheels
# Default value for now
Q = np.diag([0.03, 0.03, 0.03])
R = np.diag([0.01, 0.01, 0.01])
dt = 0.1  # time step

system = DifferentialDriveSystem(dt, lambda_, d, Q, R)
    
# Initial state and covariance
mu0 = np.array([0, 0, 0])
Sigma0 = 1000 * np.eye(3)

ekf = ExtendedKalmanFilter(
    mu0, Sigma0,
    system
)

# To store estimates
x_est = []
P_est = []

In [5]:
client = ClientAsync()
node = await client.wait_for_node()
await node.lock()
await node.wait_for_variables({"prox.horizontal"})

def motors(l_speed=0, r_speed=0):
    return {
        "motor.left.target": [int(l_speed)],
        "motor.right.target": [int(r_speed)],
    }



In [24]:
await node.set_variables(motors(50,50))

{'error_code': 2}

In [8]:
await node.set_variables(motors(0,0))

In [6]:
async def fsm():
    
    cap = cv2.VideoCapture(0) 

    print("Starting path following with obstacle avoidance...")
    for step in range(200):
        # Get camera image and process
        ret, frame = cap.read()
        if not ret:
            cap.release()
            raise RuntimeError("Impossible de capturer une frame webcam.")
        
        polys, robot_state = vision.process(
                                frame,
                                show_debug=False,    # on ne veut pas l'image debug ici
                                skip_blur=False,
                                log_pose=False
                            )

        # --- get sensors from Thymio ---
        sensor_vals = list(node.v.prox.horizontal)
        
        speed_left  = node.v.motor.left.speed
        speed_right = node.v.motor.right.speed
        u = np.array([speed_left, speed_right])
        # pose from ekf 
        pose, cov = ekf.predict(u)

        if robot_state["found"]:
            x, y = robot_state["center"]
            theta = robot_state["theta"]
            print("x:", x, "y:", y, "theta:", theta)
            z = np.array([x, y, theta])
            pose, cov = ekf.update(z)

        x_est.append(pose)
        P_est.append(cov)

        # --- update local grid ---
        grid.update_from_sensor_vals(sensor_vals)

        # --- compute virtual goal ---
        virt_world, LA_world, F_att, F_rep = navigator.compute_virtual_goal(
            pose, path, sensor_vals=sensor_vals
        )
        if virt_world is None:
            break

        # --- controller: pose -> motors ---
        uL, uR, info = g2g.compute_motor_commands(pose, virt_world)

        await node.set_variables({
            "motor.left.target":  [uL],
            "motor.right.target": [uR],
        })

        
        # small sleep
        await client.sleep(dt)

    # stop motors at the end
    await node.set_variables({
        "motor.left.target":  [0],
        "motor.right.target": [0],
    })
    await node.unlock()

In [7]:
await fsm()

Starting path following with obstacle avoidance...
x: 106 y: 532 theta: 0.029841884667109353
x: 110 y: 532 theta: 0.030293759918775143
x: 118 y: 531 theta: 0.0
x: 127 y: 530 theta: -0.029841884667109353
x: 136 y: 529 theta: -0.0596307131311693
x: 146 y: 528 theta: -0.10566558209985577
x: 156 y: 525 theta: -0.1481600207318312
x: 166 y: 523 theta: -0.19448011621510553
x: 175 y: 519 theta: -0.24135549057540384
x: 184 y: 515 theta: -0.31249155973451903
x: 194 y: 510 theta: -0.3970794452249822
x: 201 y: 504 theta: -0.47018346364673724
x: 209 y: 497 theta: -0.5757583175232357
x: 216 y: 490 theta: -0.6610431688506868
x: 222 y: 481 theta: -0.7645678433612312
x: 228 y: 472 theta: -0.8574382287842761
x: 232 y: 464 theta: -0.9302106616363873
x: 236 y: 454 theta: -1.015302591529353
x: 239 y: 445 theta: -1.1071487177940904
x: 240 y: 435 theta: -1.1902899496825317
x: 241 y: 424 theta: -1.2904966704564456
x: 241 y: 415 theta: -1.3677511095414314
x: 239 y: 404 theta: -1.4553696664279718
x: 237 y: 395 

CancelledError: 

In [25]:
await node.unlock()

{'error_code': 2}